In [2]:
# "C:\\Users\\Andrey\\Desktop\\Calc\\test_target_2.png"

In [ ]:
# -------- Two-picture hole detection + group center + clicks (OpenCV) --------
import cv2 as cv
import numpy as np
import time
import math

# ================= helpers =================
def _to_gray_norm(img):
    lab = cv.cvtColor(img, cv.COLOR_BGR2LAB)
    L, a, b = cv.split(lab)
    L = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(L)
    bgr = cv.cvtColor(cv.merge([L, a, b]), cv.COLOR_LAB2BGR)
    gray = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
    return cv.bilateralFilter(gray, 5, 50, 50)


def find_diamond_center(bgr):
    """Return center of the largest dark blob (the diamond)."""
    gray = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
    inv = cv.GaussianBlur(cv.bitwise_not(gray), (5, 5), 0)
    _, th = cv.threshold(inv, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    th = cv.morphologyEx(
        th,
        cv.MORPH_OPEN,
        cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5)),
    )
    num, _, stats, cents = cv.connectedComponentsWithStats(th, 8)
    if num <= 1:
        h, w = gray.shape[:2]
        return (w // 2, h // 2)
    idx = 1 + int(np.argmax(stats[1:, cv.CC_STAT_AREA]))
    cx, cy = cents[idx]
    return (int(cx), int(cy))


def _register_to(ref_bgr, img_bgr):
    """
    Align shot to baseline using ONLY diamond center (pure translation).
    This guarantees correct vertical/horizontal alignment of the diamond
    even if the rest of the image (holes, cropping) differs.
    """
    cx_ref, cy_ref = find_diamond_center(ref_bgr)
    cx_img, cy_img = find_diamond_center(img_bgr)

    dx = cx_ref - cx_img
    dy = cy_ref - cy_img

    H = np.array(
        [
            [1, 0, dx],
            [0, 1, dy],
            [0, 0, 1],
        ],
        dtype=np.float32,
    )

    h, w = ref_bgr.shape[:2]
    warped = cv.warpPerspective(img_bgr, H, (w, h))
    return warped, H


def _contour_filter(bin_img, min_area=150, max_area=8000, circ_min=0.2):
    cnts, _ = cv.findContours(bin_img, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    out = []
    for c in cnts:
        area = cv.contourArea(c)
        if area < min_area or area > max_area:
            continue
        peri = cv.arcLength(c, True)
        if peri <= 0:
            continue
        circ = 4 * np.pi * area / (peri * peri)
        hull = cv.convexHull(c)
        solidity = area / (cv.contourArea(hull) + 1e-6)
        if circ >= circ_min and solidity > 0.7:
            M = cv.moments(c)
            if M["m00"] != 0:
                cx, cy = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
                out.append((cx, cy, area, circ))
    return out


def estimate_group_center_from_diff_avg_robust(
    ref_gray,
    cur_gray,
    blur_ks=5,
    open_ks=3,
    close_ks=3,
    min_area=80,
    max_area=6000,
    circ_min=0.15,
    aim_xy=None,
    max_dist_px=None,
    use_area_weight=True,
):
    """
    Detect individual holes from abs-diff and return their (weighted) average center.
    Fallback: if no holes pass the filters (e.g., one big merged blob), return the
    intensity-weighted centroid of the dominant foreground component.
    Returns: (gcx, gcy, r_eq, union_mask, diff_img, holes)
    """
    diff = cv.absdiff(ref_gray, cur_gray)
    diff_blur = (
        cv.GaussianBlur(diff, (blur_ks, blur_ks), 0)
        if blur_ks and blur_ks >= 3
        else diff
    )

    # Threshold (Otsu; fallback to fixed if almost empty)
    _, th = cv.threshold(diff_blur, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    if th.mean() < 1:  # nearly no FG; try fixed small threshold
        _, th = cv.threshold(diff_blur, 25, 255, cv.THRESH_BINARY)

    # Clean edges, but DON'T dilate (we want separate contours when possible)
    if open_ks and open_ks > 1:
        th = cv.morphologyEx(
            th,
            cv.MORPH_OPEN,
            cv.getStructuringElement(cv.MORPH_ELLIPSE, (open_ks, open_ks)),
        )
    if close_ks and close_ks > 1:
        th = cv.morphologyEx(
            th,
            cv.MORPH_CLOSE,
            cv.getStructuringElement(cv.MORPH_ELLIPSE, (close_ks, close_ks)),
        )

    cnts, _ = cv.findContours(th, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    holes = []
    union = np.zeros_like(th, np.uint8)

    ax, ay = aim_xy if aim_xy is not None else (None, None)
    max_d2 = None if max_dist_px is None else (max_dist_px * max_dist_px)

    # --- Try per-hole detection ---
    for c in cnts:
        area = cv.contourArea(c)
        if area < min_area or area > max_area:
            continue
        peri = cv.arcLength(c, True)
        if peri <= 0:
            continue
        circ = 4 * np.pi * area / (peri * peri)
        if circ < circ_min:
            continue
        M = cv.moments(c)
        if M["m00"] == 0:
            continue
        cx = M["m10"] / M["m00"]
        cy = M["m01"] / M["m00"]
        if ax is not None and max_d2 is not None:
            if (cx - ax) * (cx - ax) + (cy - ay) * (cy - ay) > max_d2:
                continue
        holes.append((cx, cy, area, circ))
        cv.drawContours(union, [c], -1, 255, -1)

    if holes:
        w = (
            np.array([h[2] for h in holes], np.float64)
            if use_area_weight
            else np.ones(len(holes), np.float64)
        )
        xs = np.array([h[0] for h in holes], np.float64)
        ys = np.array([h[1] for h in holes], np.float64)
        gcx = float((xs * w).sum() / w.sum())
        gcy = float((ys * w).sum() / w.sum())
        total_area = float(np.sum(union > 0))
        if total_area <= 0:
            total_area = float(np.sum([h[2] for h in holes]))
        r_eq = float(np.sqrt(max(total_area, 1.0) / np.pi))
        return gcx, gcy, r_eq, union, diff, holes

    # --- Fallback: single blob centroid (merged hits) ---
    num, labels, stats, cents = cv.connectedComponentsWithStats(
        th, connectivity=8
    )
    if num <= 1:
        return None, None, None, th, diff, []  # truly nothing

    cand_ids = list(range(1, num))
    if aim_xy is not None:
        # pick component closest to aim
        ax, ay = aim_xy
        d2, idx = min(
            (
                (cents[i][0] - ax) ** 2 + (cents[i][1] - ay) ** 2,
                i,
            )
            for i in cand_ids
        )
    else:
        # largest area
        idx = 1 + int(np.argmax(stats[1:, cv.CC_STAT_AREA]))

    mask = (labels == idx).astype(np.uint8)
    m = cv.moments(diff.astype(np.float32) * mask, binaryImage=False)
    if m["m00"] <= 1e-6:
        # fall back to geometric centroid
        cx, cy = cents[idx]
    else:
        cx, cy = (m["m10"] / m["m00"], m["m01"] / m["m00"])

    area = float(stats[idx, cv.CC_STAT_AREA])
    r_eq = float(np.sqrt(max(area, 1.0) / np.pi))
    return (
        float(cx),
        float(cy),
        float(r_eq),
        (mask * 255).astype(np.uint8),
        diff,
        [],
    )


def _format_clicks(wind, elev):
    # wind, elev are from pixels_to_clicks above
    dir_x = "LEFT"  if wind > 0 else "RIGHT"
    dir_y = "UP"    if elev > 0 else "DOWN"
    return dir_x, dir_y, abs(wind), abs(elev)




def _overlay_click_banner(img, wind, elev):
    """Large, centered banner at the top with clicks."""
    dir_x, dir_y, wx, ey = _format_clicks(wind, elev)
    text = f"{wx:.1f} clicks {dir_x}   |   {ey:.1f} clicks {dir_y}"

    h, w = img.shape[:2]
    pad_y = 12
    bar_h = 54
    overlay = img.copy()
    cv.rectangle(overlay, (0, 0), (w, bar_h + 2 * pad_y), (0, 0, 0), -1)
    cv.addWeighted(overlay, 0.55, img, 0.45, 0, img)
    cv.putText(
        img,
        text,
        (20, bar_h),
        cv.FONT_HERSHEY_SIMPLEX,
        1.0,
        (255, 255, 255),
        2,
        cv.LINE_AA,
    )


def _draw_aim_to_group_arrow(img, aim_xy, group_xy, color=(0, 255, 255)):
    if aim_xy is None or group_xy is None:
        return
    a = (int(aim_xy[0]), int(aim_xy[1]))
    g = (int(group_xy[0]), int(group_xy[1]))
    cv.arrowedLine(img, a, g, color, 2, cv.LINE_AA, tipLength=0.03)


def _resize_to_box(img, max_w=1500, max_h=1000):
    H, W = img.shape[:2]
    s = min(max_w / max(W, 1), max_h / max(H, 1))
    if s >= 1.0:
        return img.copy()  # already small enough; keep original
    return cv.resize(img, (int(W * s), int(H * s)), interpolation=cv.INTER_AREA)


def show_operator_view(
    base_img,
    wind,
    elev,
    win_name="Zeroing â€” Operator View",
    fullscreen=False,
    save_key="s",
    max_w=1500,
    max_h=1000,
):
    img = base_img.copy()
    _overlay_click_banner(img, wind, elev)

    # upscale/downscale for a bigger default window
    disp = _resize_to_box(img, max_w=max_w, max_h=max_h)

    cv.namedWindow(win_name, cv.WINDOW_NORMAL)
    if fullscreen:
        cv.setWindowProperty(
            win_name, cv.WND_PROP_FULLSCREEN, cv.WINDOW_FULLSCREEN
        )

    cv.imshow(win_name, disp)
    while True:
        k = cv.waitKey(0) & 0xFF
        if k in (ord("q"), 27):
            break
        if k == ord(save_key):
            fn = f"zeroing_{int(time.time())}.png"
            cv.imwrite(fn, disp)
            print(f"[saved] {fn}")
    cv.destroyWindow(win_name)

def pixels_to_clicks(dx_px, dy_px, px_per_cm, distance_m, click_value_cm=0.5):
    """
    Convert dx, dy offsets (pixels) -> correction in clicks.

    dx_px, dy_px: group_center - aim_center  (image coords)
      +dx = group to the RIGHT of aim
      +dy = group BELOW aim

    px_per_cm: pixels per centimeter on target
    click_value_cm: cm shift on target per 1 click (here 0.5 cm)

    Returns:
      wind, elev  (floats)
      wind > 0  -> need clicks LEFT
      wind < 0  -> need clicks RIGHT
      elev > 0  -> need clicks UP
      elev < 0  -> need clicks DOWN
    """

    # pixel -> centimeters (offset of group from aim)
    dx_cm = dx_px / px_per_cm      # +: group is RIGHT
    dy_cm = dy_px / px_per_cm      # +: group is DOWN

    # offset in clicks (no sign flip; sign encodes direction)
    wind = dx_cm / click_value_cm  # +: RIGHT of aim  -> "LEFT" correction
    elev = dy_cm / click_value_cm  # +: BELOW aim     -> "UP" correction

    return wind, elev



def _overlay_click_box(img, wind, elev, pos=(18, 28)):
    """Draw a semi-transparent box with click instructions."""
    dir_x, dir_y, wx, ey = _format_clicks(wind, elev)
    lines = [
        f"Windage: {wx:.1f} clicks {dir_x}",
        f"Elevation: {ey:.1f} clicks {dir_y}",
    ]
    # compute box
    pad, lh = 8, 22
    w = (
        max(
            cv.getTextSize(
                s, cv.FONT_HERSHEY_SIMPLEX, 0.6, 1
            )[0][0]
            for s in lines
        )
        + 2 * pad
    )
    h = len(lines) * lh + 2 * pad
    x0, y0 = pos
    x1, y1 = x0 + w, y0 + h

    # alpha rectangle
    overlay = img.copy()
    cv.rectangle(overlay, (x0, y0), (x1, y1), (20, 20, 20), -1)
    cv.addWeighted(overlay, 0.55, img, 0.45, 0, img)

    # text
    y = y0 + pad + 16
    for s in lines:
        cv.putText(
            img,
            s,
            (x0 + pad, y),
            cv.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            1,
            cv.LINE_AA,
        )
        y += lh


def _draw_correction_arrow(img, from_xy, to_xy, color=(0, 200, 0)):
    """
    Draw an arrow from current group center (from_xy) to desired aim point (to_xy).
    Default color: green.
    """
    if from_xy is None or to_xy is None:
        return
    a = (int(from_xy[0]), int(from_xy[1]))  # group
    b = (int(to_xy[0]), int(to_xy[1]))  # aim
    cv.arrowedLine(img, a, b, color, 3, cv.LINE_AA, tipLength=0.035)


def estimate_px_per_cm_from_grid(gray, grid_size_cm=1.0):
    g = cv.GaussianBlur(gray, (3, 3), 0)
    edges = cv.Canny(g, 60, 120)
    edges = cv.dilate(
        edges, cv.getStructuringElement(cv.MORPH_RECT, (3, 3)), 1
    )
    px_x = _period_from_profile(edges.sum(axis=0))
    px_y = _period_from_profile(edges.sum(axis=1))
    vals = [v for v in (px_x, px_y) if v is not None]
    return (float(np.mean(vals)) / grid_size_cm) if vals else 40.0


def _period_from_profile(p):
    p = p.astype(np.float32)
    p = (p - p.mean()) / (p.std() + 1e-6)
    ac = np.correlate(p, p, mode="full")[len(p) - 1 :]
    min_lag = 6
    if min_lag >= len(ac):
        return None
    return min_lag + int(np.argmax(ac[min_lag:]))


# ---- one-window panel ----
def _to_bgr(img):
    return cv.cvtColor(img, cv.COLOR_GRAY2BGR) if img.ndim == 2 else img


def _titled(img, title):
    out = img.copy()
    cv.rectangle(out, (0, 0), (out.shape[1], 24), (30, 30, 30), -1)
    cv.putText(
        out,
        title,
        (8, 18),
        cv.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv.LINE_AA,
    )
    return out


def _fit_into(img, target_wh):
    w, h = target_wh
    H, W = img.shape[:2]
    s = min(w / max(W, 1), h / max(H, 1))
    im = cv.resize(
        img, (max(1, int(W * s)), max(1, int(H * s))), cv.INTER_AREA
    )
    canvas = np.full((h, w, 3), 45, np.uint8)
    y0 = (h - im.shape[0]) // 2
    x0 = (w - im.shape[1]) // 2
    canvas[y0 : y0 + im.shape[0], x0 : x0 + im.shape[1]] = im
    return canvas


def _panel(named, cell=(420, 420), cols=4, gap=8, bg=(18, 18, 18)):
    w, h = cell
    tiles = []
    for t, im in named:
        b = (
            np.full((h, w, 3), 45, np.uint8)
            if im is None
            else _fit_into(_to_bgr(im), (w, h))
        )
        tiles.append(_titled(b, t))
    r = (len(tiles) + cols - 1) // cols
    th, tw = tiles[0].shape[:2]
    panel = np.full(
        ((th + gap) * r + gap, (tw + gap) * cols + gap, 3),
        bg,
        np.uint8,
    )
    i = 0
    for R in range(r):
        for C in range(cols):
            if i >= len(tiles):
                break
            y = gap + R * (th + gap)
            x = gap + C * (tw + gap)
            panel[y : y + th, x : x + tw] = tiles[i]
            i += 1
    return panel


def _draw_holes_scaled(bgr, holes, color=(0, 255, 0)):
    vis = bgr.copy()
    for (x, y, area, _) in holes:
        r = max(4, int(np.sqrt(area / np.pi)))
        cv.circle(vis, (x, y), r, color, 2)
        cv.circle(vis, (x, y), max(2, r // 3), color, -1)
    return vis








# ================= main (two pictures) =================
if __name__ == "__main__":
    BASELINE_PATH = r"C:\Users\Andrey\Desktop\Calc\t4.png"
    SHOT_PATH = r"C:\Users\Andrey\Desktop\Calc\t4_two_clusters.png"
    DISTANCE_M = 50.0
    CLICK_MRAD = 0.1
    GRID_CM = 1.0

    baseline = cv.imread(BASELINE_PATH)
    assert baseline is not None
    shot_raw = cv.imread(SHOT_PATH)
    assert shot_raw is not None

    # register shot to baseline (diamond-center alignment)
    shot_warped, _ = _register_to(baseline, shot_raw)

    base_gray = _to_gray_norm(baseline)
    shot_gray = _to_gray_norm(shot_warped)

    # Aim point (diamond center) in warped shot / baseline space
    aim_px = find_diamond_center(shot_warped)

    # --- debug candidate holes (diff + morphology) ---
    diff = cv.absdiff(base_gray, shot_gray)
    inv = cv.bitwise_not(shot_gray)
    tophat = cv.morphologyEx(
        inv,
        cv.MORPH_TOPHAT,
        cv.getStructuringElement(cv.MORPH_ELLIPSE, (9, 9)),
    )
    mix = cv.addWeighted(diff, 0.6, tophat, 0.4, 0)

    th = cv.adaptiveThreshold(
        mix,
        255,
        cv.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv.THRESH_BINARY,
        31,
        2,
    )
    th = cv.medianBlur(th, 5)
    th = cv.morphologyEx(
        th,
        cv.MORPH_OPEN,
        cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5)),
    )
    th = cv.morphologyEx(
        th,
        cv.MORPH_CLOSE,
        cv.getStructuringElement(cv.MORPH_ELLIPSE, (7, 7)),
    )
    holes_dbg = _contour_filter(th, min_area=150, max_area=8000, circ_min=0.2)

    # --- robust group center from abs diff ---
    gcx, gcy, r_eq, group_mask, diff_s, holes = (
        estimate_group_center_from_diff_avg_robust(
            base_gray,
            shot_gray,
            blur_ks=5,
            open_ks=3,
            close_ks=3,
            min_area=80,
            max_area=6000,
            circ_min=0.15,
            aim_xy=aim_px,      # gate by diamond center
            max_dist_px=None,   # set e.g. 300â€“400 px if needed
            use_area_weight=True,
        )
    )

    # calibration: pixels per cm from CURRENT target grid
    px_per_cm = estimate_px_per_cm_from_grid(shot_gray, GRID_CM)

    # --- compute clicks ---
    if gcx is not None:
        dx, dy = gcx - aim_px[0], gcy - aim_px[1]
        wind, elev = pixels_to_clicks(dx, dy, px_per_cm, DISTANCE_M)


        print(
            f"[two-pic] group center=({gcx:.1f},{gcy:.1f}); "
            f"windage {wind:+.1f} clicks, elevation {elev:+.1f} clicks"
        )
    else:
        print("[two-pic] group center not found.")
        wind = elev = 0.0

    # ---- One-window debug overlay ----
    SHOW_INDIVIDUAL_HOLES = False
    if SHOW_INDIVIDUAL_HOLES:
        overlay = _draw_holes_scaled(shot_warped, holes, (0, 255, 0))
    else:
        overlay = shot_warped.copy()

    if gcx is not None:
        cv.circle(
            overlay,
            (int(gcx), int(gcy)),
            max(6, int(r_eq)),
            (0, 255, 255),
            2,
        )  # yellow = group
    cv.drawMarker(
        overlay, aim_px, (255, 255, 0), cv.MARKER_CROSS, 18, 2
    )  # cyan = aim

    if gcx is not None:
        _draw_aim_to_group_arrow(overlay, aim_px, (gcx, gcy))
        _overlay_click_box(overlay, wind, elev, pos=(18, 28))

    # ---- Operator overlay (clean) ----
    op = shot_warped.copy()
    if gcx is not None:
        cv.circle(
            op,
            (int(gcx), int(gcy)),
            max(6, int(r_eq)),
            (0, 255, 255),
            2,
        )
    cv.drawMarker(op, aim_px, (255, 255, 0), cv.MARKER_CROSS, 18, 2)

    if gcx is not None:
        _draw_correction_arrow(op, (gcx, gcy), aim_px, color=(0, 200, 0))
        show_operator_view(op, wind, elev, fullscreen=False, max_w=1500, max_h=1000)

    panel = _panel(
        [
            ("Warped + Group/Aim", overlay),
            ("Baseline Gray", base_gray),
            ("Shot Gray", shot_gray),
            ("Abs Diff", diff),
            ("Top-hat (inv L)", tophat),
            ("Mix (Diff+Tophat)", mix),
            ("Binary Mask", th),
            ("Group Mask", group_mask),
        ],
        cell=(420, 420),
        cols=4,
        gap=8,
    )

    cv.imshow("Two-picture Hole Detection â€“ Panel", panel)
    cv.waitKey(0)
    cv.destroyAllWindows()


[two-pic] group center=(586.1,902.3); windage +6.8 clicks, elevation +15.1 clicks


In [1]:
# -------- Two-picture hole detection + group center + clicks (OpenCV) --------
import cv2 as cv
import numpy as np
import time
import math

# ================= helpers =================
def _to_gray_norm(img):
    lab = cv.cvtColor(img, cv.COLOR_BGR2LAB)
    L, a, b = cv.split(lab)
    L = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(L)
    bgr = cv.cvtColor(cv.merge([L, a, b]), cv.COLOR_LAB2BGR)
    gray = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
    return cv.bilateralFilter(gray, 5, 50, 50)


def find_target_center(bgr, template=None):
    """Return center of the target aiming mark.

    template=None  ->  largest-dark-blob heuristic (works for any solid dark
                       shape: diamond, circle, square, filled cross, ...).
    template=<bgr> ->  normalised template matching (works for ANY shape,
                       including hollow rings, crosshairs, coloured marks).
                       Cropped automatically by set_clean_target().
    """
    if template is not None:
        t_gray = cv.cvtColor(template, cv.COLOR_BGR2GRAY) if template.ndim == 3 else template
        i_gray = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
        res = cv.matchTemplate(i_gray, t_gray, cv.TM_CCOEFF_NORMED)
        _, _, _, max_loc = cv.minMaxLoc(res)
        th, tw = t_gray.shape[:2]
        return (int(max_loc[0] + tw // 2), int(max_loc[1] + th // 2))

    # --- fallback: largest dark blob ---
    gray = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
    inv = cv.GaussianBlur(cv.bitwise_not(gray), (5, 5), 0)
    _, th = cv.threshold(inv, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    th = cv.morphologyEx(
        th,
        cv.MORPH_OPEN,
        cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5)),
    )
    num, _, stats, cents = cv.connectedComponentsWithStats(th, 8)
    if num <= 1:
        h, w = gray.shape[:2]
        return (w // 2, h // 2)
    idx = 1 + int(np.argmax(stats[1:, cv.CC_STAT_AREA]))
    cx, cy = cents[idx]
    return (int(cx), int(cy))


find_diamond_center = find_target_center  # backward-compat alias


def _register_to(ref_bgr, img_bgr, center_template=None):
    """
    Align shot to baseline using ONLY target center (pure translation).
    This guarantees correct vertical/horizontal alignment of the aiming mark
    even if the rest of the image (holes, cropping) differs.
    """
    cx_ref, cy_ref = find_target_center(ref_bgr, center_template)
    cx_img, cy_img = find_target_center(img_bgr, center_template)

    dx = cx_ref - cx_img
    dy = cy_ref - cy_img

    H = np.array(
        [
            [1, 0, dx],
            [0, 1, dy],
            [0, 0, 1],
        ],
        dtype=np.float32,
    )

    h, w = ref_bgr.shape[:2]
    warped = cv.warpPerspective(img_bgr, H, (w, h), borderValue=(255, 255, 255))
    return warped, H


def _contour_filter(bin_img, min_area=150, max_area=8000, circ_min=0.2):
    cnts, _ = cv.findContours(bin_img, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    out = []
    for c in cnts:
        area = cv.contourArea(c)
        if area < min_area or area > max_area:
            continue
        peri = cv.arcLength(c, True)
        if peri <= 0:
            continue
        circ = 4 * np.pi * area / (peri * peri)
        hull = cv.convexHull(c)
        solidity = area / (cv.contourArea(hull) + 1e-6)
        if circ >= circ_min and solidity > 0.7:
            M = cv.moments(c)
            if M["m00"] != 0:
                cx, cy = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
                out.append((cx, cy, area, circ))
    return out


def estimate_group_center_from_diff_avg_robust(
    ref_gray,
    cur_gray,
    blur_ks=5,
    open_ks=3,
    close_ks=3,
    min_area=80,
    max_area=6000,
    circ_min=0.15,
    aim_xy=None,
    max_dist_px=None,
    use_area_weight=True,
):
    """
    Detect individual holes from abs-diff and return their (weighted) average center.
    Fallback: if no holes pass the filters (e.g., one big merged blob), return the
    intensity-weighted centroid of the dominant foreground component.
    Returns: (gcx, gcy, r_eq, union_mask, diff_img, holes)
    """
    diff = cv.absdiff(ref_gray, cur_gray)
    diff_blur = (
        cv.GaussianBlur(diff, (blur_ks, blur_ks), 0)
        if blur_ks and blur_ks >= 3
        else diff
    )

    # Threshold (Otsu; fallback to fixed if almost empty)
    _, th = cv.threshold(diff_blur, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    if th.mean() < 1:  # nearly no FG; try fixed small threshold
        _, th = cv.threshold(diff_blur, 25, 255, cv.THRESH_BINARY)

    # Clean edges, but DON'T dilate (we want separate contours when possible)
    if open_ks and open_ks > 1:
        th = cv.morphologyEx(
            th,
            cv.MORPH_OPEN,
            cv.getStructuringElement(cv.MORPH_ELLIPSE, (open_ks, open_ks)),
        )
    if close_ks and close_ks > 1:
        th = cv.morphologyEx(
            th,
            cv.MORPH_CLOSE,
            cv.getStructuringElement(cv.MORPH_ELLIPSE, (close_ks, close_ks)),
        )

    cnts, _ = cv.findContours(th, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    holes = []
    union = np.zeros_like(th, np.uint8)

    ax, ay = aim_xy if aim_xy is not None else (None, None)
    max_d2 = None if max_dist_px is None else (max_dist_px * max_dist_px)

    # --- Try per-hole detection ---
    for c in cnts:
        area = cv.contourArea(c)
        if area < min_area or area > max_area:
            continue
        peri = cv.arcLength(c, True)
        if peri <= 0:
            continue
        circ = 4 * np.pi * area / (peri * peri)
        if circ < circ_min:
            continue
        M = cv.moments(c)
        if M["m00"] == 0:
            continue
        cx = M["m10"] / M["m00"]
        cy = M["m01"] / M["m00"]
        if ax is not None and max_d2 is not None:
            if (cx - ax) * (cx - ax) + (cy - ay) * (cy - ay) > max_d2:
                continue
        holes.append((cx, cy, area, circ))
        cv.drawContours(union, [c], -1, 255, -1)

    if holes:
        w = (
            np.array([h[2] for h in holes], np.float64)
            if use_area_weight
            else np.ones(len(holes), np.float64)
        )
        xs = np.array([h[0] for h in holes], np.float64)
        ys = np.array([h[1] for h in holes], np.float64)
        gcx = float((xs * w).sum() / w.sum())
        gcy = float((ys * w).sum() / w.sum())
        total_area = float(np.sum(union > 0))
        if total_area <= 0:
            total_area = float(np.sum([h[2] for h in holes]))
        r_eq = float(np.sqrt(max(total_area, 1.0) / np.pi))
        return gcx, gcy, r_eq, union, diff, holes

    # --- Fallback: single blob centroid (merged hits) ---
    num, labels, stats, cents = cv.connectedComponentsWithStats(
        th, connectivity=8
    )
    if num <= 1:
        return None, None, None, th, diff, []  # truly nothing

    cand_ids = list(range(1, num))
    if aim_xy is not None:
        # pick component closest to aim
        ax, ay = aim_xy
        d2, idx = min(
            (
                (cents[i][0] - ax) ** 2 + (cents[i][1] - ay) ** 2,
                i,
            )
            for i in cand_ids
        )
    else:
        # largest area
        idx = 1 + int(np.argmax(stats[1:, cv.CC_STAT_AREA]))

    mask = (labels == idx).astype(np.uint8)
    m = cv.moments(diff.astype(np.float32) * mask, binaryImage=False)
    if m["m00"] <= 1e-6:
        # fall back to geometric centroid
        cx, cy = cents[idx]
    else:
        cx, cy = (m["m10"] / m["m00"], m["m01"] / m["m00"])

    area = float(stats[idx, cv.CC_STAT_AREA])
    r_eq = float(np.sqrt(max(area, 1.0) / np.pi))
    return (
        float(cx),
        float(cy),
        float(r_eq),
        (mask * 255).astype(np.uint8),
        diff,
        [],
    )


def _format_clicks(wind, elev):
    # wind, elev are from pixels_to_clicks above
    dir_x = "LEFT"  if wind > 0 else "RIGHT"
    dir_y = "UP"    if elev > 0 else "DOWN"
    return dir_x, dir_y, abs(wind), abs(elev)




def _overlay_click_banner(img, wind, elev):
    """Large, centered banner at the top with clicks."""
    dir_x, dir_y, wx, ey = _format_clicks(wind, elev)
    text = f"{wx:.1f} clicks {dir_x}   |   {ey:.1f} clicks {dir_y}"

    h, w = img.shape[:2]
    pad_y = 12
    bar_h = 54
    overlay = img.copy()
    cv.rectangle(overlay, (0, 0), (w, bar_h + 2 * pad_y), (0, 0, 0), -1)
    cv.addWeighted(overlay, 0.55, img, 0.45, 0, img)
    cv.putText(
        img,
        text,
        (20, bar_h),
        cv.FONT_HERSHEY_SIMPLEX,
        1.0,
        (255, 255, 255),
        2,
        cv.LINE_AA,
    )


def _draw_aim_to_group_arrow(img, aim_xy, group_xy, color=(0, 255, 255)):
    if aim_xy is None or group_xy is None:
        return
    a = (int(aim_xy[0]), int(aim_xy[1]))
    g = (int(group_xy[0]), int(group_xy[1]))
    cv.arrowedLine(img, a, g, color, 2, cv.LINE_AA, tipLength=0.03)


def _resize_to_box(img, max_w=1500, max_h=1000):
    H, W = img.shape[:2]
    s = min(max_w / max(W, 1), max_h / max(H, 1))
    if s >= 1.0:
        return img.copy()  # already small enough; keep original
    return cv.resize(img, (int(W * s), int(H * s)), interpolation=cv.INTER_AREA)


def show_operator_view(
    base_img,
    wind,
    elev,
    win_name="Zeroing â€” Operator View",
    fullscreen=False,
    save_key="s",
    max_w=1500,
    max_h=1000,
):
    img = base_img.copy()
    _overlay_click_banner(img, wind, elev)

    # upscale/downscale for a bigger default window
    disp = _resize_to_box(img, max_w=max_w, max_h=max_h)

    cv.namedWindow(win_name, cv.WINDOW_NORMAL)
    if fullscreen:
        cv.setWindowProperty(
            win_name, cv.WND_PROP_FULLSCREEN, cv.WINDOW_FULLSCREEN
        )

    cv.imshow(win_name, disp)
    while True:
        k = cv.waitKey(0) & 0xFF
        if k in (ord("q"), 27):
            break
        if k == ord(save_key):
            fn = f"zeroing_{int(time.time())}.png"
            cv.imwrite(fn, disp)
            print(f"[saved] {fn}")
    cv.destroyWindow(win_name)

def pixels_to_clicks(dx_px, dy_px, px_per_cm, distance_m, click_value_cm=0.5):
    """
    Convert dx, dy offsets (pixels) -> correction in clicks.

    dx_px, dy_px: group_center - aim_center  (image coords)
      +dx = group to the RIGHT of aim
      +dy = group BELOW aim

    px_per_cm: pixels per centimeter on target
    click_value_cm: cm shift on target per 1 click (here 0.5 cm)

    Returns:
      wind, elev  (floats)
      wind > 0  -> need clicks LEFT
      wind < 0  -> need clicks RIGHT
      elev > 0  -> need clicks UP
      elev < 0  -> need clicks DOWN
    """

    # pixel -> centimeters (offset of group from aim)
    dx_cm = dx_px / px_per_cm      # +: group is RIGHT
    dy_cm = dy_px / px_per_cm      # +: group is DOWN

    # offset in clicks (no sign flip; sign encodes direction)
    wind = dx_cm / click_value_cm  # +: RIGHT of aim  -> "LEFT" correction
    elev = dy_cm / click_value_cm  # +: BELOW aim     -> "UP" correction

    return wind, elev



def _overlay_click_box(img, wind, elev, pos=(18, 28)):
    """Draw a semi-transparent box with click instructions."""
    dir_x, dir_y, wx, ey = _format_clicks(wind, elev)
    lines = [
        f"Windage: {wx:.1f} clicks {dir_x}",
        f"Elevation: {ey:.1f} clicks {dir_y}",
    ]
    # compute box
    pad, lh = 8, 22
    w = (
        max(
            cv.getTextSize(
                s, cv.FONT_HERSHEY_SIMPLEX, 0.6, 1
            )[0][0]
            for s in lines
        )
        + 2 * pad
    )
    h = len(lines) * lh + 2 * pad
    x0, y0 = pos
    x1, y1 = x0 + w, y0 + h

    # alpha rectangle
    overlay = img.copy()
    cv.rectangle(overlay, (x0, y0), (x1, y1), (20, 20, 20), -1)
    cv.addWeighted(overlay, 0.55, img, 0.45, 0, img)

    # text
    y = y0 + pad + 16
    for s in lines:
        cv.putText(
            img,
            s,
            (x0 + pad, y),
            cv.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            1,
            cv.LINE_AA,
        )
        y += lh


def _draw_correction_arrow(img, from_xy, to_xy, color=(0, 200, 0)):
    """
    Draw an arrow from current group center (from_xy) to desired aim point (to_xy).
    Default color: green.
    """
    if from_xy is None or to_xy is None:
        return
    a = (int(from_xy[0]), int(from_xy[1]))  # group
    b = (int(to_xy[0]), int(to_xy[1]))  # aim
    cv.arrowedLine(img, a, b, color, 3, cv.LINE_AA, tipLength=0.035)


def estimate_px_per_cm_from_grid(gray, grid_size_cm=1.0):
    g = cv.GaussianBlur(gray, (3, 3), 0)
    edges = cv.Canny(g, 60, 120)
    edges = cv.dilate(
        edges, cv.getStructuringElement(cv.MORPH_RECT, (3, 3)), 1
    )
    px_x = _period_from_profile(edges.sum(axis=0))
    px_y = _period_from_profile(edges.sum(axis=1))
    vals = [v for v in (px_x, px_y) if v is not None]
    return (float(np.mean(vals)) / grid_size_cm) if vals else 40.0


def _period_from_profile(p):
    p = p.astype(np.float32)
    p = (p - p.mean()) / (p.std() + 1e-6)
    ac = np.correlate(p, p, mode="full")[len(p) - 1 :]
    min_lag = 6
    if min_lag >= len(ac):
        return None
    return min_lag + int(np.argmax(ac[min_lag:]))


# ---- one-window panel ----
def _to_bgr(img):
    return cv.cvtColor(img, cv.COLOR_GRAY2BGR) if img.ndim == 2 else img


def _titled(img, title):
    out = img.copy()
    cv.rectangle(out, (0, 0), (out.shape[1], 24), (30, 30, 30), -1)
    cv.putText(
        out,
        title,
        (8, 18),
        cv.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv.LINE_AA,
    )
    return out


def _fit_into(img, target_wh):
    w, h = target_wh
    H, W = img.shape[:2]
    s = min(w / max(W, 1), h / max(H, 1))
    im = cv.resize(
        img, (max(1, int(W * s)), max(1, int(H * s))), cv.INTER_AREA
    )
    canvas = np.full((h, w, 3), 45, np.uint8)
    y0 = (h - im.shape[0]) // 2
    x0 = (w - im.shape[1]) // 2
    canvas[y0 : y0 + im.shape[0], x0 : x0 + im.shape[1]] = im
    return canvas


def _panel(named, cell=(420, 420), cols=4, gap=8, bg=(18, 18, 18)):
    w, h = cell
    tiles = []
    for t, im in named:
        b = (
            np.full((h, w, 3), 45, np.uint8)
            if im is None
            else _fit_into(_to_bgr(im), (w, h))
        )
        tiles.append(_titled(b, t))
    r = (len(tiles) + cols - 1) // cols
    th, tw = tiles[0].shape[:2]
    panel = np.full(
        ((th + gap) * r + gap, (tw + gap) * cols + gap, 3),
        bg,
        np.uint8,
    )
    i = 0
    for R in range(r):
        for C in range(cols):
            if i >= len(tiles):
                break
            y = gap + R * (th + gap)
            x = gap + C * (tw + gap)
            panel[y : y + th, x : x + tw] = tiles[i]
            i += 1
    return panel


def _draw_holes_scaled(bgr, holes, color=(0, 255, 0)):
    vis = bgr.copy()
    for (x, y, area, _) in holes:
        r = max(4, int(np.sqrt(area / np.pi)))
        cv.circle(vis, (x, y), r, color, 2)
        cv.circle(vis, (x, y), max(2, r // 3), color, -1)
    return vis



class ZeroingSession:
    """
    Session object that keeps a moving baseline.

    Usage:
        sess = ZeroingSession(distance_m=50.0, grid_cm=1.0)

        # 1) clean target
        clean = cv.imread("clean.png")
        sess.set_clean_target(clean)

        # 2) first shooting
        img1 = cv.imread("shot1.png")
        res1 = sess.process_shot(img1, show_debug=True, show_operator=True)

        # 3) second shooting
        img2 = cv.imread("shot2.png")
        res2 = sess.process_shot(img2, show_debug=True, show_operator=True)
    """

    def __init__(self, distance_m=50.0, grid_cm=1.0, click_value_cm=0.5):
        self.distance_m = distance_m
        self.grid_cm = grid_cm
        self.click_value_cm = click_value_cm

        self.baseline_bgr = None
        self.baseline_gray = None
        self.center_template = None

        self.round_index = 0   # just for debug / logging

    # ---------- setup ----------

    def set_clean_target(self, img_bgr):
        """
        Set the baseline to the clean target image and learn the center template.
        The template is a patch cropped around the aiming mark; all subsequent
        calls use template matching so the session works with any target shape.
        """
        if img_bgr is None:
            raise ValueError("set_clean_target: img_bgr is None")

        self.baseline_bgr = img_bgr.copy()
        self.baseline_gray = _to_gray_norm(self.baseline_bgr)
        self.round_index = 0

        # Auto-learn center template (blob method, one-time on clean image).
        cx, cy = find_target_center(img_bgr)
        h, w = img_bgr.shape[:2]
        half = max(30, int(min(h, w) * 0.08))  # ~8% of shorter side
        x1, y1 = max(0, cx - half), max(0, cy - half)
        x2, y2 = min(w, cx + half), min(h, cy + half)
        self.center_template = img_bgr[y1:y2, x1:x2].copy()
        print(f"[session] baseline set — center template "
              f"{self.center_template.shape[1]}x{self.center_template.shape[0]}px "
              f"at ({cx},{cy})")

    # ---------- per-round processing ----------

    def process_shot(self,
                     shot_bgr,
                     show_debug=True,
                     show_operator=True,
                     win_prefix="Zeroing"):
        """
        Process a new shot image:
          - register it to current baseline,
          - detect new group center,
          - compute clicks,
          - return all useful info,
          - update baseline to this shot.

        Returns a dict with:
          {
            "gc": (gcx, gcy) or None,
            "aim": (ax, ay),
            "wind": wind,
            "elev": elev,
            "shot_warped": shot_warped,
            "overlay": overlay_img,
            "operator": operator_img,
          }
        """
        if self.baseline_bgr is None or self.baseline_gray is None:
            raise RuntimeError("Baseline not set. Call set_clean_target() first.")

        if shot_bgr is None:
            raise ValueError("process_shot: shot_bgr is None")

        self.round_index += 1
        print(f"\n[session] processing round #{self.round_index}")

        # 1) register shot to baseline (target-center alignment)
        shot_warped, _ = _register_to(self.baseline_bgr, shot_bgr, self.center_template)

        # 2) grayscale / normalization
        shot_gray = _to_gray_norm(shot_warped)

        # 3) aim point (target center)
        aim_px = find_target_center(shot_warped, self.center_template)

        # 4) robust group center from abs diff (NEW HOLES ONLY)
        gcx, gcy, r_eq, group_mask, diff_s, holes = \
            estimate_group_center_from_diff_avg_robust(
                self.baseline_gray,
                shot_gray,
                blur_ks=5,
                open_ks=3,
                close_ks=3,
                min_area=80,
                max_area=6000,
                circ_min=0.15,
                aim_xy=aim_px,      # gate by diamond center
                max_dist_px=None,   # or 300â€“400 if you want
                use_area_weight=True,
            )

        # 5) calibration: pixels per cm from CURRENT target grid
        px_per_cm = estimate_px_per_cm_from_grid(shot_gray, self.grid_cm)

        if gcx is not None:
            dx, dy = gcx - aim_px[0], gcy - aim_px[1]
            wind, elev = pixels_to_clicks(dx, dy, px_per_cm,
                                          self.distance_m,
                                          click_value_cm=self.click_value_cm)
            print(
                f"[session] group center=({gcx:.1f},{gcy:.1f}); "
                f"windage {wind:+.1f} clicks, elevation {elev:+.1f} clicks"
            )
        else:
            print("[session] group center not found.")
            wind = elev = 0.0

        # 6) build overlays (reuse your existing drawing helpers)
        SHOW_INDIVIDUAL_HOLES = False
        if SHOW_INDIVIDUAL_HOLES:
            overlay = _draw_holes_scaled(shot_warped, holes, (0, 255, 0))
        else:
            overlay = shot_warped.copy()

        if gcx is not None:
            cv.circle(
                overlay,
                (int(gcx), int(gcy)),
                max(6, int(r_eq)),
                (0, 255, 255),
                2,
            )  # yellow = group
        cv.drawMarker(
            overlay, aim_px, (255, 255, 0), cv.MARKER_CROSS, 18, 2
        )  # cyan = aim

        if gcx is not None:
            _draw_aim_to_group_arrow(overlay, aim_px, (gcx, gcy))
            _overlay_click_box(overlay, wind, elev, pos=(18, 28))

        # Operator overlay (clean)
        op = shot_warped.copy()
        if gcx is not None:
            cv.circle(
                op,
                (int(gcx), int(gcy)),
                max(6, int(r_eq)),
                (0, 255, 255),
                2,
            )
        cv.drawMarker(op, aim_px, (255, 255, 0), cv.MARKER_CROSS, 18, 2)
        if gcx is not None:
            _draw_correction_arrow(op, (gcx, gcy), aim_px, color=(0, 200, 0))

        # 7) show windows (optional)
        if show_operator:
    # fixed window name so OpenCV reuses the same window & size
            show_operator_view(
                op,
                wind,
                elev,
                win_name=f"{win_prefix} â€” Operator View",
                fullscreen=False,
                max_w=1500,
                max_h=1000,
            )


        if show_debug:
            # Build debug panel like your old main
            diff = cv.absdiff(self.baseline_gray, shot_gray)

            inv = cv.bitwise_not(shot_gray)
            tophat = cv.morphologyEx(
                inv,
                cv.MORPH_TOPHAT,
                cv.getStructuringElement(cv.MORPH_ELLIPSE, (9, 9)),
            )
            mix = cv.addWeighted(diff, 0.6, tophat, 0.4, 0)

            th = cv.adaptiveThreshold(
                mix,
                255,
                cv.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv.THRESH_BINARY,
                31,
                2,
            )
            th = cv.medianBlur(th, 5)
            th = cv.morphologyEx(
                th,
                cv.MORPH_OPEN,
                cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5)),
            )
            th = cv.morphologyEx(
                th,
                cv.MORPH_CLOSE,
                cv.getStructuringElement(cv.MORPH_ELLIPSE, (7, 7)),
            )

            panel = _panel(
                [
                    ("Warped + Group/Aim", overlay),
                    ("Baseline Gray", self.baseline_gray),
                    ("Shot Gray", shot_gray),
                    ("Abs Diff", diff),
                    ("Top-hat (inv L)", tophat),
                    ("Mix (Diff+Tophat)", mix),
                    ("Binary Mask", th),
                    ("Group Mask", group_mask),
                ],
                cell=(420, 420),
                cols=4,
                gap=8,
            )

            cv.imshow(f"{win_prefix} â€” Debug Panel (round {self.round_index})", panel)
            cv.waitKey(0)
            cv.destroyWindow(f"{win_prefix} â€” Debug Panel (round {self.round_index})")

        # 8) MOVE BASELINE FORWARD
        self.baseline_bgr = shot_warped.copy()
        self.baseline_gray = shot_gray.copy()

        # 9) pack and return
        if gcx is not None:
            gc = (float(gcx), float(gcy))
        else:
            gc = None

        return {
            "gc": gc,
            "aim": (float(aim_px[0]), float(aim_px[1])),
            "wind": float(wind),
            "elev": float(elev),
            "shot_warped": shot_warped,
            "overlay": overlay,
            "operator": op,
        }




# ================= main (session-based) =================
if __name__ == "__main__":
    # paths for demo:
    CLEAN_PATH = r"C:\Users\Andrey\Desktop\Calc\images\target_empty.png"                 # clean target
    SHOT1_PATH = r"C:\Users\Andrey\Desktop\Calc\images\t4.png"    # for testing: one round
    SHOT2_PATH = r"C:\Users\Andrey\Desktop\Calc\images\t4_two_clusters.png"    # for testing: second round

    clean = cv.imread(CLEAN_PATH)
    assert clean is not None, "Failed to load clean image"

    shot1 = cv.imread(SHOT1_PATH)
    assert shot1 is not None, "Failed to load first shot image"

    # Create session
    sess = ZeroingSession(distance_m=50.0, grid_cm=1.0, click_value_cm=0.5)

    # 1) set baseline
    sess.set_clean_target(clean)

    # 2) process first round (clean -> shot1)
    res1 = sess.process_shot(shot1,
                             show_debug=True,
                             show_operator=True,
                             win_prefix="Zeroing")

    # If you have a second shot image, e.g. SHOT2_PATH:
    shot2 = cv.imread(SHOT2_PATH)
    res2 = sess.process_shot(shot2, show_debug=True, show_operator=True)

        


[session] clean baseline set

[session] processing round #1
[session] group center=(450.0,796.7); windage +0.0 clicks, elevation +9.8 clicks

[session] processing round #2
[session] group center=(586.1,902.3); windage +6.8 clicks, elevation +15.1 clicks
